In [ ]:
import yaml
import camelot
import pandas as pd
import joblib
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_chroma import Chroma
from chromadb.errors import NotFoundError
from langchain_community.retrievers import BM25Retriever

# pd.set_option('display.max_rows', 500)
# pd.set_option('display.max_columns', 500)
# pd.set_option('display.width', 1000)

In [63]:
with open('config.yaml', 'rb') as yaml_file:
    config=yaml.safe_load(yaml_file)

In [3]:
pdf_filepath = 'fy2024_analysis_of_revenue_and_expenditure.pdf'
docs = PyPDFLoader(pdf_filepath).load()
splitter = RecursiveCharacterTextSplitter(
    chunk_size=config['chunk_size'], 
    chunk_overlap=config['chunk_overlap'],
    separators=['\n']
)

#TODO - switch to table detection model/method for scaling
table_pages = [8, 11, 12, 16, 19, 20, 24, 25, 26, 27, 
               28, 29, 30, 31, 32, 33, 34]

chunks = splitter.split_documents(docs)

for chunk in chunks:
    if chunk.metadata.get('total_pages') is not None:
        del chunk.metadata['total_pages']
    if chunk.metadata.get('page_label') is not None:
        chunk.metadata['page'] = int(chunk.metadata['page_label'])

non_table_chunks = [chunk for chunk in chunks 
                    if int(chunk.metadata['page_label']) not in table_pages]

In [ ]:
row_chunk_list = []
for page in table_pages:
    page_chunks = [chunk for chunk in chunks if int(chunk.metadata['page_label']) == page]
    page_metadata = page_chunks[0].metadata

    table_name = ''
    for chunk in page_chunks:
        for line in chunk.page_content.split('\n'):
            if 'table' in line.lower():
                table_name = ' '.join(line.split())

    if page == 8:
        print(table_name)

    tables = camelot.read_pdf(pdf_filepath, pages=str(page), flavor='ml')
    if len(tables) > 0:
        for table in tables:
            table.df = table.df.replace({'\n': ''}, regex=True)
            table.df = table.df.replace({'BLANK': ''}, regex=True)
            table.df = table.df.replace({'  ': ' '}, regex=True)
            table.df = table.df[~table.df.eq('').all(axis=1)]

            # document specific
            if table.df[0].values[0].strip() == '' and table.df[0].values[1].strip() == '' :
                table.df = pd.concat([
                    table.df.iloc[:2].agg(' '.join).to_frame().T,
                    table.df.iloc[2:]
                ], ignore_index=True)
            table.df.columns = table.df.iloc[0]
            table.df = table.df.drop(0).reset_index(drop=True)
            col_list = table.df.columns.tolist()
            col_list[0] = '__item'
            col_list = [' '.join(col.split()) for col in col_list]
            table.df.columns = col_list
            
            if page == 8 and 'table 1.1' in table_name.lower():
                # fix for hallucinated numbers in this specific table
                table.df = table.df.drop(34)
            # print(table.df.columns)
            # row_chunk_list = []
            for index, row in table.df.iterrows():
                check_row_values = [val for val in row[1:] if val not in ('', '-')]
                if check_row_values:
                    row_number = int(str(index))
                    row_chunk_str = f'In the {table_name} within page {page}, row {row_number}\n'
                    
                    item_name = ''

                    for index, col in enumerate(col_list):
                        if col == '__item':
                            item_name = row.iloc[index]
                            row_chunk_str += f'{item_name}\n'
                        else:
                            if row.iloc[index] not in ['-', '']:
                                # TODO - flag negative value?
                                row_chunk_str += f'{col} was {row.iloc[index]}\n'

                    # row_chunk_list.append(row_chunk_str)
                    # print(row_chunk_str)
                
                # row_chunk = '\n'.join(row_chunk_list)
                # print(row_chunk)
                # print('-----------------------------------------')
                    page_metadata['table_name'] = table_name
                    page_metadata['row_number'] = row_number
                    # page_metadata['table_df_json'] = table.df.to_json(orient='index')
                    row_chunk_doc = Document(
                        page_content=row_chunk_str,
                        metadata=page_metadata
                    )
                    row_chunk_list.append(row_chunk_doc)

Table 1.1 Fiscal Position in FY2022 and FY2023


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
The `max_size` parameter is deprecated and will be removed in v4.26. Please specify in `size['longest_edge'] instead`.
c:\zdrive\miniconda3\envs\htx_case\Lib\site-packages\torch\nn\modules\module.py:2588: UserWarning: for conv1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  module._load_from_state_dict(
c:\zdrive\miniconda3\envs\htx_case\Lib\site-packages\torch\nn\modules\batchnorm.py:141: UserWarning: for bn1.weight: copying from a non-meta par

In [6]:
all_chunks_list = non_table_chunks + row_chunk_list
len(all_chunks_list)

403

In [7]:
embeddings = HuggingFaceEmbeddings(model_name='google/embeddinggemma-300m')

In [ ]:
try:
    vector_store = Chroma(
        collection_name=config['collection_name'], 
        persist_directory=config['chroma_dir'],
        create_collection_if_not_exists=False,
    )
    vector_store.delete_collection()
    print('Existing collection dropped')
except NotFoundError:
    pass

vector_store = Chroma.from_documents(
    collection_name=config['collection_name'],
    documents=all_chunks_list,
    embedding=embeddings,
    persist_directory=config['chroma_dir'],
    collection_configuration={
        'hnsw': {
            'space': 'cosine',
            'ef_construction': 200,
            'ef_search': 20,
            'max_neighbors': 32,
        }
    },
)
print('Vector Store created')

Existing collection dropped
Vector Store created


In [48]:
chroma_vectorstore = Chroma(
    collection_name=config['collection_name'],
    persist_directory=config['chroma_dir'],
    embedding_function=embeddings,
    create_collection_if_not_exists=False
)

all_docs = chroma_vectorstore.get(include=['documents', 'metadatas'])

bm25_documents = [Document(page_content=content, metadata=meta) 
                  for content, meta in zip(all_docs['documents'], all_docs['metadatas'])]

sparse_retriever = BM25Retriever.from_documents(bm25_documents)
sparse_retriever.k = 10
with open(config['bm25_index_path'], 'wb') as bm25_file:
    joblib.dump(sparse_retriever, bm25_file)
print('BM25 Index created')

BM25 Index created


Testing/Debugging

In [ ]:
# from langchain_classic.retrievers import EnsembleRetriever

# chroma_vectorstore = Chroma(
#     collection_name=config['collection_name'],
#     persist_directory=config['chroma_dir'],
#     embedding_function=embeddings,
#     create_collection_if_not_exists=False
# )
# dense_retriever = chroma_vectorstore.as_retriever(
#     search_type='similarity',
#     search_kwargs={'k': config['retrieval_k']},
# )
# with open(config['bm25_index_path'], 'rb') as bm25_file:
#     sparse_retriever = joblib.load(bm25_file)

# hybrid_retriever = EnsembleRetriever(
#     retrievers=[dense_retriever, sparse_retriever],
#     weights=[config['vector_search_weightage'], (1-config['vector_search_weightage'])],
# )

In [ ]:
# # query = 'what is the Amount of Corporate Income Tax in 2024' # OK
# # query = 'what is the Amount of Corporate Income Tax in 2023' # OK
# # query = 'how much did corp income tax increase in 2023' # OK
# # query = 'how much will corp income tax increase in 2024' # OK
# # query = 'what is the Total amount of top ups to funds in 2023' # OK
# # query = 'what is the expected Total amount of top ups to funds in 2024' # OK
# # query = 'what is the List of taxes mentioned in the Operating Revenue section' # OK
# # query = 'what is the Latest Actual Fiscal Position in billions' # OK

# # query = "when was the document distributed?" #OK
# # query = 'what is date relating to estate duty' #OK

# # query = 'What are the key government revenue streams'
# # query = 'how will the Budget for the Future Energy Fund be supported?'
# # query = 'What are the key government revenue streams, and how will the Budget for the Future Energy Fund be supported?'

# search_results = hybrid_retriever.invoke(query)

# for result in search_results:
#     # print('metadata')
#     # print(result.metadata)
#     print('[[[content]]]')
#     print(result.page_content)
#     print('')

[[[content]]]
In the Table 2.1 Budget for FY2024 within page 16, row 30
Future Energy Fund
Estimated FY2024 $billion was 5.00


[[[content]]]
Top-ups to Endowment and Trust Funds ($20.4 billion) 
 
In Budget 2024, the Government will top up the GST Voucher Fund by               
$6.0 billion to meet the steady state cashflow needs for the enhanced 
permanent GST Voucher scheme. To support Budget 2024 enhancements to the 
Progressive Wage Credit Scheme (PWCS) and Edusave Awards, the Government 
will top up the PWCS Fund by $1.0 billion and the Edusave Endowment Fund by 
$2.0 billion respectively. The Government will top up the National Research 
Fund by $1.8 billion to support Research, Innovation and Enterprise, and the 
National Productivity Fund by $2.0 billion to boost productivity, Continuing 
Education and Training, and investment promotion. The Government will also

[[[content]]]
In the Table 2.4 Top-ups to Endowment and Trust Funds in FY2024 within page 20, row 1
Future Energy F